# Day 5 — Context engineering & caching (Gemini)
**Theory:** [day-05-context-engineering-and-caching.md](../theory/day-05-context-engineering-and-caching.md) · ⏱ ~60 min · Level: medium

**You will:**
1. Watch **implicit caching** work on a multi-turn conversation with a stable prefix
2. Break it on purpose with a timestamp ("silent cache killer")
3. Use **explicit caching** (`client.caches.create`) with a TTL
4. Trim old tool results (context hygiene), with no API needed
5. Run a mini "lost in the middle" experiment

Gemini has two caching modes:
- **Implicit:** automatic, no code. If a request starts with the same prefix as a recent one (and the prefix
  is above a minimum size), those tokens are served from cache at a discount. They show up as
  `usage_metadata.cached_content_token_count`. Implicit cache hits aren't guaranteed.
- **Explicit:** you create a cache object with a TTL and reference it. Guaranteed reuse, plus a storage cost per hour.

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
# Please don't let an AI "auto-fix" rewrite the imports below: the course packages are `course_utils` and `shopease`.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone (or update) the course repo, install it, read the key from notebook secrets
    BRANCH = os.getenv("COURSE_BRANCH", "main")
    REPO_URL = os.getenv("COURSE_REPO_URL", "https://github.com/Rahulverma5/agentic-ai-course")
    REPO = pathlib.Path(os.getenv("COURSE_BASE_DIR", "/content" if IN_COLAB else "/kaggle/working")) / "agentic-ai-course"
    if not (REPO / ".git").exists():
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    else:  # an older clone from a previous session: bring it up to date
        subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "reset", "-q", "--hard", f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt"), "-e", str(REPO)],
                   check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("GEMINI_API_KEY", userdata.get("GEMINI_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("GEMINI_API_KEY", UserSecretsClient().get_secret("GEMINI_API_KEY"))
    except Exception as e:
        print("Add GEMINI_API_KEY to the notebook secrets (🔑 icon in Colab, Add-ons → Secrets in Kaggle):", e)
else:  # Codespaces / local: use the checked-out repo
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
try:
    from google.genai import types  # Gemini SDK types: Content, Part, GenerateContentConfig, ...
    from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of, thinking, user
    from shopease.data import setup_data
except ImportError as e:
    raise ImportError(
        f"{e}\n\nThe course code at {REPO} is missing or outdated. Fix it like this:\n"
        f"  • Colab/Kaggle: run  !rm -rf {REPO}  then Runtime → Restart session, and run this cell again.\n"
        "  • Codespaces/local: run  git pull && pip install -e .  in a terminal, then restart the kernel.") from None
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup();

## 1. A realistic, long, *stable* system instruction

In [ ]:
import time, datetime, json
import pandas as pd
policies = "\n".join(p.read_text() for p in sorted(pathlib.Path("projects/shopease-support-agent/data/policies").glob("*.md")))
catalogue = "\n".join(f"- SKU{i:04}: Product {i}, ${(i * 7) % 200 + 5}.00, category {['home', 'apparel', 'electronics'][i % 3]}"
                      for i in range(600))
SYSTEM = f"## Role\nYou are ShopEase support.\n\n## Policies\n{policies}\n\n## Catalogue\n{catalogue}"
print(client.models.count_tokens(model=CHEAP_MODEL, contents=SYSTEM).total_tokens, "tokens in the prefix")

QUESTIONS = ["What's the return window?", "And for opened earbuds?", "How much is SKU0042?",
             "Do gold members pay return shipping?", "Summarise our chat in one line."]

def conversation(system_fn=lambda: SYSTEM, model=CHEAP_MODEL, cached_content=None):
    contents, rows = [], []
    for q in QUESTIONS:
        contents.append(user(q))
        cfg = types.GenerateContentConfig(max_output_tokens=300, cached_content=cached_content,
                                          system_instruction=None if cached_content else system_fn())
        t0 = time.time()
        r = client.models.generate_content(model=model, contents=contents, config=cfg)
        cost = tracker.add(model, r.usage_metadata, "cache demo")
        contents.append(r.candidates[0].content)
        u = r.usage_metadata
        rows.append({"turn": len(rows) + 1, "prompt": u.prompt_token_count, "cached": u.cached_content_token_count or 0,
                     "seconds": round(time.time() - t0, 2), "≈cost $": round(cost, 6)})
    return pd.DataFrame(rows)

## 2. Implicit caching: same prefix every turn

In [ ]:
stable = conversation()
stable

> Turn 1 has nothing cached. Later turns should show a large `cached` count, because the system instruction
> (and earlier turns) are an identical prefix. If `cached` stays 0, the prefix may be below the minimum size,
> or your model doesn't support implicit caching: make the catalogue longer, or try MODEL.

**Savings math:** saving ≈ 1 − (1 + (n−1)·r)/n on the prefix, where r is the cached-token price ratio (~0.1).

In [ ]:
for r_mult in (0.1, 0.25):
    for n in (5, 10, 20):
        print(f"r={r_mult}, n={n:>2}: saving on prefix ≈ {1 - (1 + (n - 1) * r_mult) / n:.0%}")

## 3. The silent cache killer
Putting `datetime.now()` at the start of the system instruction changes the prefix on every call.

In [ ]:
killer = conversation(system_fn=lambda: f"Current time: {datetime.datetime.now().isoformat()}\n\n{SYSTEM}")
display(killer[["turn", "prompt", "cached", "≈cost $"]])
print("Fix: put volatile data (time, user id) at the END: after the stable prefix, e.g. in the user message.")

## 4. Explicit caching: create a cache with a TTL, then reference it
Good when many requests share a big prefix (a long document, a policy manual). You pay a small
per-hour storage cost while it lives. Always delete it when you're done.

In [ ]:
try:
    cache = client.caches.create(model=CHEAP_MODEL, config=types.CreateCachedContentConfig(
        display_name="shopease-policies", system_instruction=SYSTEM, ttl="300s"))
    print("created", cache.name, "expires", cache.expire_time)
    explicit = conversation(cached_content=cache.name)
    display(explicit)
    client.caches.delete(name=cache.name)
    print("deleted", cache.name)
except Exception as e:
    print(f"Explicit caching not available for {CHEAP_MODEL} on your tier ({type(e).__name__}: {str(e)[:150]}).\n"
          "Try a pinned model id that supports caching (see the caching docs) or rely on implicit caching.")

## 5. Context hygiene: trim old tool results (no API needed)
In Gemini's format, tool results are `function_response` parts inside `user` turns.

In [ ]:
from course_utils import function_response

def trim_old_tool_results(contents, keep_last=2, placeholder="[old tool result removed]"):
    seen = 0
    for content in reversed(contents):
        for part in content.parts or []:
            if part.function_response:
                seen += 1
                if seen > keep_last:
                    part.function_response.response = {"output": placeholder}
    return contents

fake = []
for i in range(5):
    call = types.FunctionCall(id=f"c{i}", name="read_logs", args={})
    fake += [types.Content(role="model", parts=[types.Part(function_call=call)]),
             types.Content(role="user", parts=[function_response(call, "x" * 5000)])]
size = lambda cs: sum(len(c.model_dump_json()) for c in cs)
before = size(fake)
print(f"history size: {before:,} → {size(trim_old_tool_results(fake)):,} characters")

## 6. Mini "lost in the middle" experiment
Hide one fact at the start, middle or end of ~15k tokens of filler and ask about it.
(Modern models are much better at this than 2023 models, so you may see no difference. That is a finding too.)

In [ ]:
filler = [f"Note {i}: the warehouse in zone {i % 9} restocked {i * 3} boxes of item {i % 50}." for i in range(900)]
FACT = "IMPORTANT: The secret discount code for Asha is MANGO-73."
def needle_test(position):
    lines = filler.copy()
    idx = {"start": 0, "middle": len(lines) // 2, "end": len(lines)}[position]
    lines.insert(idx, FACT)
    out = ask("\n".join(lines) + "\n\nWhat is Asha's secret discount code? Reply with the code only.", max_tokens=50)
    return "MANGO-73" in out

pd.DataFrame([{"position": p, "correct (3 tries)": sum(needle_test(p) for _ in range(3))} for p in ("start", "middle", "end")])

## ✏️ Exercises
1. Run `conversation(model=MODEL)`. Do cache hits and savings differ from CHEAP_MODEL?
2. Move the timestamp to the *end* of the system instruction. Does implicit caching work again? Why only partly?
3. Put the fact in the middle of 60k tokens instead of 15k. Any change?

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker` (estimated at paid prices; the free tier costs $0). Make a habit of checking tokens and cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None